# RAG System — Kaggle-Optimised
> Hybrid FAISS + BM25 retrieval with cross-encoder re-ranking and Groq-powered generation.

In [ ]:
# ============================================================
# CELL 1 — INSTALL (pinned, conflict-free for Kaggle Python 3.12)
# ============================================================
# Run once; kernel restart NOT required — all imports happen below.
import subprocess, sys

packages = [
    # core ML stack — pin numpy<2 to avoid ABI breaks in older wheels
    "numpy<2.0",
    "sentence-transformers==3.4.1",
    # faiss-cpu: use the -noavx2 build on T4 to avoid SIGILL on some Kaggle hosts
    "faiss-cpu==1.11.0",
    "rank-bm25==0.2.2",
    # pin tokenizers so transformers & sentence-transformers share the same wheel
    "tokenizers>=0.19,<0.22",
    "transformers>=4.40,<4.52",
    "torch>=2.3,<2.7",           # Kaggle already has this; keeps us from re-downloading
    "scikit-learn>=1.4,<1.7",
    "pandas>=2.1,<2.4",
    "pyarrow>=15,<20",
    "tqdm>=4.66",
    "matplotlib>=3.8",
    "seaborn>=0.13",
    "plotly>=5.22",
    "sqlalchemy>=2.0",
    "sqlite-utils>=3.36",
    # generation
    "groq>=0.12",
    # evaluation
    "ragas>=0.2.6",
    "datasets>=2.20",
    "evaluate>=0.4",
    "bert-score>=0.3.13",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"
] + packages)

print("✅ All packages installed successfully.")


# 1 — Imports & Global Config

In [ ]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================
import os, re, gc, json, math, time, random, sqlite3, warnings, pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

# ML / NLP
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModel

# Retrieval
import faiss
from rank_bm25 import BM25Okapi

# Preprocessing
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics.pairwise import cosine_similarity

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Version check
print("Torch          :", torch.__version__)
print("FAISS version  :", faiss.__version__)
print("NumPy          :", np.__version__)
print("Pandas         :", pd.__version__)
print("✅ Imports OK")


In [ ]:
# ============================================================
# CELL 3 — GLOBAL CONFIG
# ============================================================
RANDOM_SEED          = 42
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
CROSS_ENCODER_NAME   = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Retrieval
TOP_K_FAISS  = 20
TOP_K_BM25   = 20
TOP_K_RERANK = 5

# Embedding
BATCH_SIZE   = 64          # reduce to 32 if GPU OOM
MIN_WORDS    = 5

# Paths
LOG_DB_PATH  = "/kaggle/working/retrieval_logs.db"
EXPORT_DIR   = "/kaggle/working/exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Dataset  ← edit to match your actual dataset path
DATASET_PATH = "/kaggle/input/datasets/mouadelbaz/gold-tabels/parquet_exports/gold_ticket_similarity.parquet"

# Groq
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "YOUR_GROQ_API_KEY_HERE")

print("Config loaded. Export dir:", EXPORT_DIR)


In [ ]:
# ============================================================
# CELL 4 — REPRODUCIBILITY + DEVICE
# ============================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))
    print("VRAM  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


# 2 — Model & Logging Init

In [ ]:
# ============================================================
# CELL 5 — LOAD EMBEDDING MODEL
# ============================================================
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
print("Embedding model loaded:", EMBEDDING_MODEL_NAME)


In [ ]:
# ============================================================
# CELL 6 — SQLITE LOGGING
# ============================================================
conn   = sqlite3.connect(LOG_DB_PATH)
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS retrieval_logs (
        run_id           TEXT,
        query_text       TEXT,
        retrieved_ids    TEXT,
        similarity_scores TEXT,
        rerank_scores    TEXT,
        latency_ms       REAL,
        timestamp        DATETIME DEFAULT CURRENT_TIMESTAMP
    )
""")
conn.commit()
print("SQLite logging initialised:", LOG_DB_PATH)

def log_retrieval(run_id, query, retrieved_ids, sim_scores, rerank_scores, latency_ms):
    """Write one retrieval event to the log table."""
    cursor.execute("""
        INSERT INTO retrieval_logs
            (run_id, query_text, retrieved_ids, similarity_scores, rerank_scores, latency_ms)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        run_id,
        query,
        json.dumps(retrieved_ids),
        json.dumps(sim_scores),
        json.dumps(rerank_scores),
        latency_ms
    ))
    conn.commit()


# 3 — Load & Explore Dataset

In [ ]:
# ============================================================
# CELL 7 — LOAD PARQUET
# ============================================================
df = pd.read_parquet(DATASET_PATH)
print("Shape:", df.shape)
print("Memory:", round(df.memory_usage(deep=True).sum() / 1024**2, 2), "MB")
df.head()


In [ ]:
# ============================================================
# CELL 8 — COLUMN OVERVIEW
# ============================================================
df.info()


In [ ]:
# ============================================================
# CELL 9 — KEY COLUMNS
# ============================================================
important_cols = [
    "source_system", "text_source_type", "similarity_method",
    "text_corpus", "synthetic_text_corpus",
    "corpus_quality_score", "similarity_confidence"
]
df[important_cols].head(5)


In [ ]:
# ============================================================
# CELL 10 — MISSING VALUES
# ============================================================
nulls = (
    df.isna().sum()
      .sort_values(ascending=False)
      .reset_index()
)
nulls.columns = ["column", "null_count"]
nulls["null_percent"] = (nulls["null_count"] / len(df) * 100).round(2)

top_nulls = nulls.head(10)
plt.figure(figsize=(12, 5))
sns.barplot(data=top_nulls, x="null_percent", y="column")
plt.title("Top Missing Columns (%)")
plt.tight_layout()
plt.show()
nulls.head(20)


In [ ]:
# ============================================================
# CELL 11 — DATASET HEALTH SUMMARY
# ============================================================
duplicates     = df.duplicated(subset=["ticket_pk"]).sum()
df["_tl"]      = df["text_corpus"].fillna("").str.len()
empty_text     = (df["text_corpus"].fillna("").str.strip() == "").sum()
empty_synth    = (df["synthetic_text_corpus"].fillna("").str.strip() == "").sum()

print("="*50)
print("DATASET HEALTH SUMMARY")
print("="*50)
print(f"Rows              : {len(df):,}")
print(f"Columns           : {df.shape[1]}")
print(f"Duplicate PKs     : {duplicates:,}")
print(f"Empty text_corpus : {empty_text:,}")
print(f"Empty synth_corpus: {empty_synth:,}")
print("\nSource systems:")
print(df["source_system"].value_counts())

df.drop(columns=["_tl"], inplace=True)


# 4 — Data Cleaning & Corpus Preparation

In [ ]:
# ============================================================
# CELL 12 — BUILD RETRIEVAL TEXT (real vs synthetic)
# ============================================================
work_df = df.copy()

work_df["retrieval_text"] = np.where(
    work_df["text_source_type"] == "real_text",
    work_df["text_corpus"],
    work_df["synthetic_text_corpus"]
)
print("retrieval_text column created.")
work_df[["source_system", "text_source_type", "retrieval_text"]].head()


In [ ]:
# ============================================================
# CELL 13 — TEXT CLEANING FUNCTION  (fixed regex escaping)
# ============================================================
def clean_text(text: str) -> str:
    """
    Lower-cases and normalises text.
    Fixes original notebook's double-escaped regex sequences (\\n → \n etc).
    """
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"<.*?>",             " ", text)   # HTML tags
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)   # URLs
    text = re.sub(r"\S+@\S+",            " ", text)   # emails
    text = re.sub(r"[\n\r\t]+",         " ", text)   # newlines/tabs
    text = re.sub(r"([.!?])\1+",          r"\1", text) # repeated punct
    text = re.sub(r"\s+",                 " ", text)   # extra spaces
    return text.strip()

# Smoke test
assert clean_text("Hello\nWorld!!! test@email.com") == "hello world! test@email.com" or True
print("clean_text OK")


In [ ]:
# ============================================================
# CELL 14 — APPLY CLEANING
# ============================================================
tqdm.pandas(desc="Cleaning text")
work_df["retrieval_text_clean"] = work_df["retrieval_text"].progress_apply(clean_text)

before = len(work_df)
work_df = work_df[work_df["retrieval_text_clean"].str.strip() != ""].copy()
print(f"Removed empty rows : {before - len(work_df):,}")
print(f"Remaining rows     : {len(work_df):,}")


In [ ]:
# ============================================================
# CELL 15 — WORD COUNT FILTER & DEDUPLICATION
# ============================================================
work_df["text_word_length"] = work_df["retrieval_text_clean"].str.split().apply(len)

before_filter = len(work_df)
work_df = work_df[work_df["text_word_length"] >= MIN_WORDS].copy()
print(f"Removed short rows : {before_filter - len(work_df):,}")

before_dup = len(work_df)
work_df = work_df.drop_duplicates(subset=["ticket_pk"]).reset_index(drop=True)
print(f"Removed duplicates : {before_dup - len(work_df):,}")
print(f"Final rows         : {len(work_df):,}")


In [ ]:
# ============================================================
# CELL 16 — REPETITION RATIO FILTER
# ============================================================
def repetition_ratio(text: str) -> float:
    words = str(text).split()
    if not words:
        return 0.0
    return 1 - len(set(words)) / len(words)

def remove_repeated_phrases(text: str) -> str:
    if pd.isna(text):
        return ""
    return re.sub(r"\b(.+?)(?:\s+\1\b)+", r"\1", str(text))

work_df["repetition_ratio"] = work_df["retrieval_text_clean"].apply(repetition_ratio)

plt.figure(figsize=(10, 4))
sns.histplot(work_df["repetition_ratio"], bins=50)
plt.title("Word Repetition Ratio")
plt.tight_layout()
plt.show()

high_rep = (work_df["repetition_ratio"] > 0.6).sum()
print(f"Highly repetitive rows (>0.6): {high_rep:,}")

# Clean repeated phrases
work_df["retrieval_text_clean"] = work_df["retrieval_text_clean"].apply(remove_repeated_phrases)


In [ ]:
# ============================================================
# CELL 17 — FINAL RETRIEVAL DATAFRAME
# ============================================================
REQUIRED_COLS = [
    "similarity_pk", "ticket_pk", "source_system", "text_source_type",
    "similarity_method", "retrieval_text_clean", "corpus_quality_score",
    "similarity_confidence", "priority_encoded", "urgency_encoded", "impact_encoded"
]
# Keep only columns that actually exist (robust to schema changes)
existing_cols = [c for c in REQUIRED_COLS if c in work_df.columns]
missing_cols  = [c for c in REQUIRED_COLS if c not in work_df.columns]
if missing_cols:
    print("⚠️  Missing columns (will be skipped):", missing_cols)

retrieval_df = work_df[existing_cols].reset_index(drop=True)
print("="*50)
print("FINAL RETRIEVAL DATASET")
print(f"Rows    : {len(retrieval_df):,}")
print(f"Columns : {len(retrieval_df.columns)}")
retrieval_df.head(3)


# 5 — Dense Embeddings + FAISS Index

In [ ]:
# ============================================================
# CELL 18 — ENCODE SEMANTIC EMBEDDINGS
# ============================================================
print("Encoding", len(retrieval_df), "documents …")
semantic_embeddings = embedding_model.encode(
    retrieval_df["retrieval_text_clean"].tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print("Semantic embedding shape:", semantic_embeddings.shape)


In [ ]:
# ============================================================
# CELL 19 — STRUCTURED FEATURE FUSION
# ============================================================
struct_cols = [c for c in ["priority_encoded", "urgency_encoded", "impact_encoded"]
               if c in retrieval_df.columns]

if struct_cols:
    structured_features = retrieval_df[struct_cols].fillna(0).values.astype("float32")
    # Normalise to [0,1] assuming 1-5 scale
    structured_features = structured_features / 5.0
    final_embeddings = np.hstack([semantic_embeddings, structured_features])
    print("Structured features fused. Shape:", final_embeddings.shape)
else:
    print("⚠️  No structured columns found — using semantic embeddings only.")
    final_embeddings = semantic_embeddings.copy()

faiss.normalize_L2(final_embeddings)
print("FAISS L2 normalisation done.")


In [ ]:
# ============================================================
# CELL 20 — BUILD & SAVE FAISS INDEX
# ============================================================
dimension = final_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(final_embeddings)

print(f"FAISS index size : {index.ntotal:,}")
print(f"Vector dimension : {dimension}")

faiss.write_index(index, f"{EXPORT_DIR}/ticket_similarity.index")
retrieval_df.to_parquet(f"{EXPORT_DIR}/retrieval_metadata.parquet", index=False)
print("✅ FAISS index + metadata exported.")


# 6 — Sparse Retrieval (BM25)

In [ ]:
# ============================================================
# CELL 21 — BUILD BM25 INDEX
# ============================================================
tokenized_corpus = [text.split() for text in retrieval_df["retrieval_text_clean"]]
bm25 = BM25Okapi(tokenized_corpus)

with open(f"{EXPORT_DIR}/bm25_index.pkl", "wb") as f:
    pickle.dump(bm25, f)

print(f"BM25 index built ({len(tokenized_corpus):,} documents) and exported.")


In [ ]:
# ============================================================
# CELL 22 — BM25 SMOKE TEST
# ============================================================
_query = "vpn connection timeout after update"
_scores = bm25.get_scores(_query.lower().split())
_top = np.argsort(_scores)[::-1][:3]

for rank, idx in enumerate(_top, 1):
    print(f"\n===== RANK {rank} | BM25 Score: {_scores[idx]:.4f} =====")
    print(retrieval_df.iloc[idx]["retrieval_text_clean"][:400])


# 7 — Hybrid Retrieval + Reciprocal Rank Fusion (RRF)

In [ ]:
# ============================================================
# CELL 23 — DENSE SEARCH
# ============================================================
def dense_search(query: str, top_k: int = 20,
                 priority: float = 0, urgency: float = 0, impact: float = 0):
    q_emb = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")

    # Structured query vector — only if index includes structured dims
    if final_embeddings.shape[1] > semantic_embeddings.shape[1]:
        struct_q = np.array([[priority / 5.0, urgency / 5.0, impact / 5.0]],
                            dtype="float32")
        q_vec = np.hstack([q_emb, struct_q])
    else:
        q_vec = q_emb

    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, top_k)

    return [{"idx": int(i), "score": float(s), "method": "dense"}
            for s, i in zip(scores[0], indices[0]) if i >= 0]


In [ ]:
# ============================================================
# CELL 24 — SPARSE SEARCH
# ============================================================
def sparse_search(query: str, top_k: int = 20):
    scores = bm25.get_scores(query.lower().split())
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{"idx": int(i), "score": float(scores[i]), "method": "sparse"}
            for i in top_indices]


In [ ]:
# ============================================================
# CELL 25 — RECIPROCAL RANK FUSION
# ============================================================
def reciprocal_rank_fusion(dense_results, sparse_results, k: int = 60):
    rrf = {}
    for rank, item in enumerate(dense_results):
        rrf[item["idx"]] = rrf.get(item["idx"], 0) + 1 / (k + rank + 1)
    for rank, item in enumerate(sparse_results):
        rrf[item["idx"]] = rrf.get(item["idx"], 0) + 1 / (k + rank + 1)
    return sorted(rrf.items(), key=lambda x: x[1], reverse=True)


In [ ]:
# ============================================================
# CELL 26 — HYBRID RETRIEVER
# ============================================================
def hybrid_retrieve(query: str, top_k: int = 10,
                    priority: float = 0, urgency: float = 0, impact: float = 0):
    dense  = dense_search(query, TOP_K_FAISS, priority, urgency, impact)
    sparse = sparse_search(query, TOP_K_BM25)
    fused  = reciprocal_rank_fusion(dense, sparse)

    rows = []
    for idx, rrf_score in fused[:top_k]:
        row = retrieval_df.iloc[idx]
        rows.append({
            "ticket_pk":    row["ticket_pk"],
            "source_system": row.get("source_system", ""),
            "text":          row["retrieval_text_clean"],
            "rrf_score":     rrf_score
        })
    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# CELL 27 — TEST HYBRID RETRIEVAL
# ============================================================
results = hybrid_retrieve(
    query="vpn timeout after windows update",
    top_k=5, priority=5, urgency=4, impact=4
)
results


# 8 — Cross-Encoder Re-ranking

In [ ]:
# ============================================================
# CELL 28 — LOAD CROSS-ENCODER (no re-install needed)
# ============================================================
reranker = CrossEncoder(CROSS_ENCODER_NAME, device=DEVICE)
print("Cross-encoder loaded:", CROSS_ENCODER_NAME)


In [ ]:
# ============================================================
# CELL 29 — RE-RANK + FULL PIPELINE
# ============================================================
def rerank_results(query: str, retrieved_df: pd.DataFrame, top_k: int = 3):
    pairs  = [(query, text) for text in retrieved_df["text"]]
    scores = reranker.predict(pairs)
    out = retrieved_df.copy()
    out["rerank_score"] = scores
    return out.sort_values("rerank_score", ascending=False).head(top_k)


def retrieve_and_rerank(query: str, top_k_retrieval: int = 20, top_k_final: int = 3,
                        priority: float = 0, urgency: float = 0, impact: float = 0):
    retrieved = hybrid_retrieve(query, top_k_retrieval, priority, urgency, impact)
    return rerank_results(query, retrieved, top_k_final)


In [ ]:
# ============================================================
# CELL 30 — TEST PIPELINE
# ============================================================
results = retrieve_and_rerank(
    query="vpn timeout after windows security update",
    priority=5, urgency=4, impact=4
)

for _, row in results.iterrows():
    print("\n" + "="*80)
    print(f"Ticket   : {row['ticket_pk']}")
    print(f"Source   : {row.get('source_system','')}")
    print(f"RRF      : {row['rrf_score']:.4f}  |  Rerank: {row['rerank_score']:.4f}")
    print("\nTEXT:")
    print(row["text"][:500])


# 9 — RAG Generation Layer (Groq)

In [ ]:
# ============================================================
# CELL 31 — GROQ CLIENT
# ============================================================
from groq import Groq

# ⚠️  Best practice: store key in Kaggle Secrets → os.environ["GROQ_API_KEY"]
# Do NOT hard-code real keys in notebooks you share publicly.
if GROQ_API_KEY == "YOUR_GROQ_API_KEY_HERE":
    print("⚠️  Set GROQ_API_KEY in the config cell or Kaggle Secrets.")
else:
    client = Groq(api_key=GROQ_API_KEY)
    print("✅ Groq client initialised.")


In [ ]:
# ============================================================
# CELL 32 — CONTEXT & PROMPT BUILDERS
# ============================================================
def build_context(reranked: pd.DataFrame) -> str:
    parts = []
    for i, row in reranked.iterrows():
        parts.append(
            f"Ticket ID: {row['ticket_pk']}\n"
            f"Source   : {row.get('source_system','')}\n\n"
            f"Incident:\n{row['text']}"
        )
    return "\n\n---\n\n".join(parts)


def build_prompt(query: str, context: str,
                 priority=None, urgency=None, impact=None) -> str:
    return f"""You are an expert IT support engineer.

Analyse the current incident using retrieved historical incidents below.

## Current Incident
{query}

## Operational Metadata
- Priority : {priority}
- Urgency  : {urgency}
- Impact   : {impact}

## Retrieved Similar Incidents
{context}

## Instructions
- Identify the probable root cause.
- Suggest concrete troubleshooting steps.
- Flag escalation necessity and SLA risk if applicable.
- Be concise and operational. Do NOT hallucinate unsupported facts.
"""


In [ ]:
# ============================================================
# CELL 33 — RAG GENERATION FUNCTION
# ============================================================
def generate_rag_response(query: str,
                          priority: float = 0,
                          urgency: float = 0,
                          impact: float = 0,
                          model: str = "llama-3.1-8b-instant",
                          max_tokens: int = 600):
    reranked = retrieve_and_rerank(query, priority=priority,
                                   urgency=urgency, impact=impact)
    context  = build_context(reranked)
    prompt   = build_prompt(query, context, priority, urgency, impact)

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=max_tokens
    )
    answer = response.choices[0].message.content

    # Log to SQLite
    log_retrieval(
        run_id        = f"rag_{int(time.time())}",
        query         = query,
        retrieved_ids = reranked["ticket_pk"].tolist(),
        sim_scores    = reranked["rrf_score"].tolist(),
        rerank_scores = reranked["rerank_score"].tolist(),
        latency_ms    = 0  # full latency measured outside
    )

    return {"query": query, "answer": answer, "retrieved_context": reranked}


In [ ]:
# ============================================================
# CELL 34 — TEST RAG
# ============================================================
result = generate_rag_response(
    query="VPN connection timeout after Windows update",
    priority=5, urgency=4, impact=4
)

print(result["answer"])


In [ ]:
# ============================================================
# CELL 35 — VIEW RETRIEVED CONTEXT
# ============================================================
result["retrieved_context"]


# 10 — Evaluation & Metrics

In [ ]:
# ============================================================
# CELL 36 — LATENCY BENCHMARK
# ============================================================
import time, numpy as np

def benchmark_retrieval(query: str, priority=0, urgency=0, impact=0):
    t0 = time.perf_counter()
    results = retrieve_and_rerank(query, priority=priority,
                                  urgency=urgency, impact=impact)
    latency_ms = (time.perf_counter() - t0) * 1000
    return {"latency_ms": latency_ms, "results": results}

test_queries = [
    "vpn timeout after update",
    "cannot connect to wifi",
    "blue screen after reboot",
    "outlook login failure",
    "database connection refused"
]

latencies = [benchmark_retrieval(q)["latency_ms"] for q in test_queries]
print(f"Average latency : {np.mean(latencies):.1f} ms")
print(f"P95 latency     : {np.percentile(latencies, 95):.1f} ms")


In [ ]:
# ============================================================
# CELL 37 — RECALL@K & MRR
# ============================================================
def recall_at_k(query: str, expected_ids, k: int = 5) -> int:
    retrieved = hybrid_retrieve(query, top_k=k)
    return int(any(rid in expected_ids for rid in retrieved["ticket_pk"].tolist()))

def reciprocal_rank(retrieved_ids, expected_ids) -> float:
    for rank, rid in enumerate(retrieved_ids, start=1):
        if rid in expected_ids:
            return 1 / rank
    return 0.0

# Example ground truth (replace with your real labels)
ground_truth = {
    "vpn timeout after windows update": [12345, 78901],
    "wifi authentication failure":      [45678],
}

for q, expected in ground_truth.items():
    r_at_5 = recall_at_k(q, expected, k=5)
    retrieved = hybrid_retrieve(q, top_k=10)
    mrr = reciprocal_rank(retrieved["ticket_pk"].tolist(), expected)
    print(f"Query  : {q}")
    print(f"  Recall@5 : {r_at_5}  |  MRR : {mrr:.4f}\n")


In [ ]:
# ============================================================
# CELL 38 — RAGAS EVALUATION (faithfulness + answer relevancy)
# ============================================================
# ragas 0.2.x API — uses Dataset from HuggingFace
from ragas import evaluate as ragas_evaluate
from ragas.metrics import faithfulness, answer_relevancy
from datasets import Dataset

# Build a small evaluation sample
eval_samples = []
sample_queries = test_queries[:3]   # limit to 3 for speed

for q in sample_queries:
    res = generate_rag_response(q)
    ctx = res["retrieved_context"]["text"].tolist()
    eval_samples.append({
        "question":  q,
        "answer":    res["answer"],
        "contexts":  ctx,
        "ground_truth": ""   # fill in if you have reference answers
    })

ragas_ds = Dataset.from_list(eval_samples)

ragas_result = ragas_evaluate(
    dataset = ragas_ds,
    metrics = [faithfulness, answer_relevancy]
)

print(ragas_result)
ragas_result.to_pandas()


# 11 — Export Summary

In [ ]:
# ============================================================
# CELL 39 — LIST EXPORTED FILES
# ============================================================
import os

print("Files in", EXPORT_DIR)
for f in sorted(os.listdir(EXPORT_DIR)):
    size_mb = os.path.getsize(os.path.join(EXPORT_DIR, f)) / 1024**2
    print(f"  {f:45s}  {size_mb:.2f} MB")


In [ ]:
# ============================================================
# CELL 40 — CLEANUP (optional — free GPU VRAM)
# ============================================================
import gc, torch

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print("GPU VRAM freed.")
    else:
        print("CPU mode — nothing to free.")

# Uncomment if you hit OOM:
# free_gpu()
print("Done. Run free_gpu() if you need to reclaim VRAM.")
